## Builtin Tools in langchain

### 1. DuckDuckGo search tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

#Here tools are also runnables, so they have their own invoke function
results = search_tool.invoke('AI news')

print(results)

AI News is an umbrella term for the continuously updated informational ecosystem reporting on artificial intelligence developments, including research results, model releases, product launches, corporate strategy, regulation, labor impacts, security… 2 days ago - AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide. 8 hours ago - News coverage on artificial intelligence and machine learning tech, the companies building them, and the ethical issues AI raises today. 2 days ago - Artificial Intelligence: Read latest updates on AI like Google AI, ChatGPT, Google Lamda, Bard chatbot and more along with latest news as AI technology advances and makes new progress. All get detailed articles on AI related queries like what is AI, types of artificial intelligence, its applications and future. 12 hours ago - An overview of Google’s new global partnerships and funding announcements at the AI Impact Summit in In

### 2. Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('whoami')

print(results)

Executing command:
 whoami
dragon



/home/dragon/.virtualenvs/langchain-campusX/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [7]:
results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
Tools-in-langchain.ipynb



## Custom Tools in langchain

### Method 1- Using tool decorartor

In [8]:
from langchain_core.tools import tool

#### Step 1: Create a function

Here docstring is very important because LLM will understand the purpose of this function only with the help of this doc-string

In [9]:
def multiply(a,b):
    """ Multiply two numbers """
    return a*b 

#### Step 2: Add Type Hints

Here, while defining the function, we will drop the hints so that LLM will know which data type to send, this is not enforced in python , but still its a suggestion

In [10]:
def multiply(a: int, b: int) -> int:
    """ Multiply two numbers """
    return a*b 

#### Step 3: Add tool decorator

Now, this decorator is what makes this function accessible to the LLM, so the entire magic lies in this decorator only

In [28]:
@tool
def multiply(a: int, b: int)-> int:
    """ Multiply two numbers """
    return a*b 

#### Now, use the tool


Now, this tool is also a runnable in langchain, so we need to use invoke function to get the output.


In [13]:
result = multiply.invoke(
    {"a": 2, "b": 3}
)

print(result)

6


#### Lets explore the properties of this tool


Now, every tool has these 3 attributes mentioned below.

In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


This is also true for inbuilt tools

In [15]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


#### Now, when we send this tool to an LLM, lets look at what it actually sees

In [16]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers ', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### Method 2 - Using Structured Tool

First define the pydantic data-class and the function which we want to use as tool

In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required= True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a*b

/tmp/ipykernel_169032/1418027348.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required= True, description="The first number to add")
/tmp/ipykernel_169032/1418027348.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


Now, we define the tool, by combining them both
1. The function is passed
2. The pydantic class defined above will be passed as the arguments schema 

In [18]:
multiply_tool = StructuredTool.from_function(
    func = multiply_func, # The function is passed here
    name = "multiply",
    description= "Multiply two numbers",
    args_schema=MultiplyInput # The pydantic class which we defined above is passed as the arguments schema
)

Now, we will call the tool

In [21]:
result = multiply_tool.invoke({"a": 3, "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


Now, lets check if it really enforces the constraints or not by passing string values instead of integers.

There are 2 scenarios here:
1. If the passed string can be converted to an integer, then it will do it.
2. If it cannot be converted to an integer, it will throw an error

In [24]:
result = multiply_tool.invoke({"a": "3", "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


Now, above we passed a number in the form of a string , it worked, but now lets pass "-" and see if it throws an error

In [25]:
result = multiply_tool.invoke({"a": "-", "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

ValidationError: 1 validation error for MultiplyInput
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='-', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

Now, lets test the same thing on the tool defined with @tool decorator

So, apparently @tool is also updated to enforce the rules just like pydantic, its not like previous langchain versions

In [26]:
result = multiply.invoke({"a": "-", "b" : 4})
print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)

ValidationError: 1 validation error for multiply
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='-', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing